# 05 · Document Parsing：结构化解析

> 加载是“读进来”，解析是“读懂结构”。尤其 PDF 不能只做 `pdf→text`——那样表格、图片、层级全丢了。

**本文件覆盖知识点**：文本提取 / 表格提取 / 图片提取 / 标题识别 / 章节识别 / 页码 / Header-Footer / 文档结构恢复 / OCR / Layout Analysis

目标态：
```text
PDF ──> 文本 + 表格 + 图片 + 版面结构(标题/章节/页码)   ← 而不是纯 text
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 为什么要“结构化”解析

同一份 PDF，两种解析结果：

```text
差:  "价格表¥998¥1288产品名基础版专业版"      # 表格横竖错位，读不出字段
好:  [表格] 产品名=基础版 价格=¥998            # 字段可检索、可过滤
    [图片] 架构图.png（单独存，供多模态检索）
    [标题] 第2节 价格体系（可用于层级切分/元数据）
```

保留结构，后面 **切分（07）**、**元数据（09）**、**多模态检索（30）** 才有料可用。

In [ ]:
# 知识点·真调说明：结构化解析的价值 —— 同一定价信息，“纯文本串”答不准，“结构化表格”能逐行作答
print('① 纯文本抽取：表格被压成几串文字，行列对应关系全丢')
_llm_live(
    prompt='某价格页经“纯文本抽取”(未做表格识别)后，各单元格被挤成三串，原行列关系已丢失：\n'
           '串1：998元/月  1999元/月  免费试用\n'
           '串2：基础版  专业版  企业版\n'
           '串3：公有云  支持私有化  定制\n'
           '（三串之间是否一一对应、对应顺序如何，抽取时已不可考）\n'
           '【问题】专业版一个月多少钱？它支持私有化部署吗？',
    system='你是售前顾问，只能依据上面原文判断；对应关系在原文里丢了就如实说明“无法确定”，不要强行配对猜测。',
    fallback='纯文本无法可靠作答：998/1999 与 基础/专业/企业 之间的一一对应在抽取时已丢失，'
             '可能 998 是专业版也可能 1999 才是；串3 也看不出“支持私有化”是哪个档位的能力——只能答“无法确定”。',
    temperature=0.2,
)
print()
print('② 结构化解析：同一信息还原成带行列的表，能定位、能逐行引用')
_llm_live(
    prompt='【结构化表格】\n'
           '| 档位 | 价格 | 部署方式 |\n'
           '| 基础版 | 998 元/月 | 公有云 |\n'
           '| 专业版 | 1999 元/月 | 公有云 + 支持私有化 |\n'
           '| 企业版 | 免费试用 | 定制部署 |\n'
           '【问题】专业版一个月多少钱？它支持私有化部署吗？请注明依据的是第几行。',
    system='你是售前顾问，按表格逐行作答，并注明依据行号。',
    fallback='专业版 1999 元/月，部署方式为“公有云 + 支持私有化”——依据表格第 2 行（专业版）。',
    temperature=0.2,
)
print()
print('同样的信息：纯文本只能靠猜/答“无法确定”，结构化表格则能确定并“按行引用”。')
print('→ 这就是 Parsing 要保留表格结构的原因：下游的检索与问答，需要的是“一一对应的字段”而不是一串无从拆分的字。')

In [ ]:
# 快速看：PyMuPDF 至少能给出“按页”的文本与基本块信息
import fitz  # PyMuPDF

def pdf_to_structured_pages(pdf_path: str) -> list:
    """逐页提取文本，记录页码；这是结构化解析的起点"""
    pages = []
    with fitz.open(pdf_path) as doc:
        for pno in range(len(doc)):
            page = doc[pno]
            text = page.get_text()
            # blocks 可近似看作“版面块”：文字块/图片块，能帮我们保留粗结构
            n_blocks = len(page.get_text('blocks'))
            pages.append({
                'page': pno + 1,
                'text': text,
                'blocks': n_blocks,
                'has_image': bool(page.get_images()),
            })
    return pages

print('PDF 结构化解析函数已就绪。')
print('提示: 把任意 PDF 放入 data/ 后，调用 pdf_to_structured_pages 即可看到 页/块/图片 信息。')

## 2. 解析金字塔（自底向上）

| 层次 | 提取内容 | 产出 |
|------|---------|------|
| 版面(Layout) | 文字块/图片块坐标 | Layout Analysis：分栏还原阅读顺序 |
| 结构 | 标题层级/章节/页码 | 供层级切分与元数据 |
| 对象 | 表格(行列表头)、图片、公式 | 表格结构化、图片另存 |
| 像素 | 扫描件文字 | OCR 识别成文本再走上面流程 |

常见工具：
- 轻量：**PyMuPDF**(block/坐标)、**PDFPlumber**(表格定位)；
- 生产：**Unstructured**(partition_pdf)、**Docling**(版面+表格)、**MinerU**(中文/数学公式强)；
- OCR：**PaddleOCR** / **RapidOCR**(中文好)、Tesseract(通用)。

## 3. OCR：扫描件怎么进 RAG

```text
扫描图片 ──> OCR(版面检测+文字识别) ──> 带坐标的文本 ──> 表格还原/结构解析 ──> 向量库
```

OCR 的常见坑：
- 表格会被 OCR 成“无空格”的文字，需要**版面模型**还原行列；
- 双栏文档需要按栏还原顺序，否则正文前后颠倒；
- 繁体/竖排/公式需要专门的模型。



In [ ]:
# 知识点·真调说明：OCR 后处理 —— 扫描件转出的字带错，交给模型按上下文做“词级纠错”
_llm_live(
    prompt='下面是某扫描件经 OCR 后的原文，存在空格被吞、形近字写错、断句错位等问题：\n'
           '【OCR原文】本产 品支寺公有云SaaS 与私 有化部暑，答不上来的问题会自动转接人 工，并携带完 整的对话上下文。\n'
           '请把它还原成规范的一句话：只修 OCR 引入的错字/断句，不要增删或改写原文意思；'
           '第二行用“<错字→改正>”列表说明你改了哪几处。',
    system='你是 OCR 后处理助手，只做识别纠错与断句，不添加原文没有的信息。输出两行：纠错后的句子；<错字→改正>列表。',
    fallback='纠错后：本产品支持公有云 SaaS 与私有化部署，答不上来的问题会自动转接人工，并携带完整的对话上下文。\n'
             '<支寺→支持>  <部暑→部署>  <转接人 工→转接人工>  <完 整→完整>',
    temperature=0.2,
)
print('→ OCR 把“像素”变成“文字”，但常夹带错字与粘连；让 LLM 按上下文做词级纠错是常见的 OCR 后处理手段——'
      '再往前才是 06 课的规则化清洗（控制字符 / 零宽 / 空白规整），两道关卡一起保证进库文本干净。')

## 小结

- Parsing 的目标是 **文本+表格+图片+结构** 四件套；
- 按复杂度选工具：轻量 PyMuPDF / 生产 Unstructured·Docling·MinerU；
- 扫描件先 OCR。

解析出的长文本仍不能直接向量化：先做**数据清洗（06）**，再把它切成 chunk（07）。